# Document Extraction Engine: Kaggle demo

Upload a document and the engine will:

1. **Split** it into page images at 200 DPI.
2. **Classify** the document from page 1: category (structured, unstructured or handwritten), document type, language and rotation.
3. **Extract** the fields from each page with the model for that category.
4. **Draw** a box on each value, and build one JSON result.

**Before you start:** in the right panel, open **Session options** and set **Internet: ON** and **Accelerator: GPU T4 x2**. Then run the one code cell. Full instructions are in `docs/kaggle/README.md`.

## How the pipeline works

**1. Split the upload into page images.** A PDF is rendered at 200 DPI, one image per page. An image file is used as it is. Each page is cleaned up (contrast, deskew).

**2. Classify the document once, from page 1.** A small copy of page 1 goes to the router model (Qwen3-VL-4B). It returns:
- **category:** structured, unstructured or handwritten
- **document type:** DOC-01, DOC-02 or DOC-03 when a schema matches, otherwise generic
- **sub-type** for unstructured documents (brochure, pamphlet, menu card, and so on)
- **language** and **rotation**. Upright pages are turned upright before extraction.

The same result is applied to every page. You can override any of these by hand.

**3. Route each page by category.**

| Category | What reads the page | Boxes? |
| --- | --- | --- |
| Handwritten (including hand-filled forms) | Qwen3-VL-4B reads the page image | Yes |
| Structured, digital (typed or printed) | GLM-OCR reads the text, then Qwen3-4B-Instruct turns the text into JSON | No, values are flagged for review |
| Unstructured, digital | Same as structured, digital | No |

**4. Extract the fields.** The model returns each field's value and, where it can, a box as `[x1, y1, x2, y2]` on a 0–1000 scale. The app converts the box to pixels on the page. A fixed schema (DOC-01 to DOC-03) is used only when you pick the type. Otherwise the fields are named from the labels on the page.

**5. Check and merge.** Values for schema fields are normalised and checked (PAN, dates, amounts). Every page's fields are merged into one JSON with page numbers. Values without a box, failed checks and handwriting are flagged `review_required`.

**6. Build the JSON and draw the boxes.** The result has the document metadata, one entry per field with its value and evidence, the flags, and a record of which models ran. Each page is drawn with its boxes in the **Visual Grounding** tab.

**What is not yet measured:** accuracy on your documents, and how well the GLM-OCR route handles digital forms. Confidence is not calibrated, so every document is marked `needs_review`.


## Run this one cell

This single cell does everything: gets the latest code, installs what is missing (first run only, a few minutes), and launches the app with a public link. **Needs Internet ON and a GPU (T4) in the session options.**

Handwritten pages are read in one pass by Qwen3-VL-4B at up to 2 MP (`DOC_MAX_IMAGE_PIXELS`), so small handwriting keeps more detail. If you get `CUDA out of memory`, change the 2000000 in the cell to 1300000 and re-run.

In [ ]:
# ONE CELL: get the latest code, install anything missing, launch the app. Re-run after every code change.
import os, subprocess, sys
os.chdir("/kaggle/working")
REPO = "/kaggle/working/DOC_Extractor_QUAPT"
URL = "https://github.com/dharicharan/DOC_Extractor_QUAPT.git"
BRANCH = "feat/page-read"

def sh(*cmd, **kw):
    return subprocess.run(list(cmd), capture_output=True, text=True, **kw)

if not os.path.isdir(REPO + "/.git"):
    r = sh("git", "clone", "-b", BRANCH, URL, REPO)
    if r.returncode != 0:
        raise SystemExit("Clone failed: turn Internet ON in the session options and re-run. " + r.stderr)
else:
    sh("git", "-C", REPO, "fetch", "origin", BRANCH)
    sh("git", "-C", REPO, "checkout", "-B", BRANCH, "origin/" + BRANCH)
print("Running commit:", sh("git", "-C", REPO, "log", "-1", "--oneline").stdout)

# Skip the install when the packages are already there. find_spec looks them up without importing them,
# so nothing heavy (pandas, torch) is loaded into the notebook kernel; app.py runs as its own process.
import importlib.util
needed = ["transformers", "qwen_vl_utils", "gradio", "bitsandbytes", "cv2", "pypdfium2", "instructor", "litellm"]
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("Installing dependencies (first run only), missing:", missing)
    get_ipython().system("pip install -q uv")
    get_ipython().system(f"cd {REPO} && uv pip install --system -r requirements.txt && uv pip install --system -e .")
    get_ipython().system("uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio")
else:
    print("Dependencies already installed.")
print("Starting the app: the model loads now; wait for the gradio.live link (a few minutes).")

os.environ.setdefault("DOC_MAX_IMAGE_PIXELS", "2000000")  # page read at up to 2 MP; 1300000 if CUDA OOM
subprocess.run("fuser -k 7860/tcp 2>/dev/null || true", shell=True)
os.chdir(REPO + "/src/doc_extractor")
get_ipython().system("python app.py")

## Using the app

1. Upload a PDF, JPG, PNG or TIFF.
2. Leave the overrides on **Auto-Detect** to classify automatically.
3. Click **Classify & Extract** and watch the **Pipeline steps** panel.
4. Read the **Visual Grounding** tab (boxes on each page) and the **Canonical JSON Output** tab (download available).

**Expected behaviour**
- The first classification is slower, because the model loads then.
- Every page of a PDF is read, one page at a time.
- The digital route (GLM-OCR + text model) gives no boxes, so its values are withheld: `value: null`, `null_reason: EVIDENCE_MISSING`.
- Each extracted field has a `confidence` from the model's token probabilities; below 0.75 it is marked for review.

## Troubleshooting

| Problem | Fix |
| --- | --- |
| `Temporary failure in name resolution` or "Could not resolve host" | Turn Internet ON, restart the session, re-run the cell. |
| `No such file or directory` in the cell | The clone failed. Turn Internet ON and re-run the cell. |
| `CUDA out of memory` | Set `%env DOC_MAX_IMAGE_PIXELS=1000000` above the `os.chdir` line in the cell, restart, re-run the cell. |
| Steps panel shows an old result or no "filled by" text | Check the commit printed by the cell and re-run the cell. |
| GLM-OCR fails to load | Run `!uv pip install --system -U transformers`, restart, re-run the cell. |

Send the steps table, the status line and the commit printed by the cell when asking for help.